In [2]:
!pip -q install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.3 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

set_seed(42)

### Load raw dataset splits

In [4]:
sft_train_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_sft")
sft_val_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_sft")

dpo_train_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
dpo_val_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs")


print("Raw SFT Train:", len(sft_train_raw))
print("Raw SFT Val  :", len(sft_val_raw))
print("Raw DPO Train:", len(dpo_train_raw))
print("Raw DPO Val  :", len(dpo_val_raw))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Raw SFT Train: 61135
Raw SFT Val  : 1000
Raw DPO Train: 61135
Raw DPO Val  : 2000


_Make smaller splits and helper function_

In [6]:
sft_train_raw = sft_train_raw.shuffle(seed=42).select(range(3000))
sft_val_raw = sft_val_raw.select(range(200))

dpo_train_raw = dpo_train_raw.shuffle(seed=42).select(range(3000))
dpo_val_raw = dpo_val_raw.select(range(200))

print("SFT Train:", len(sft_train_raw))
print("SFT Val  :", len(sft_val_raw))
print("DPO Train:", len(dpo_train_raw))
print("DPO Val  :", len(dpo_val_raw))

SFT Train: 3000
SFT Val  : 200
DPO Train: 3000
DPO Val  : 200


_Inspect one raw SFT sample and one raw DPO sample_

In [7]:
def get_last_assistant_text(messages):
    for message in reversed(messages):
        if message["role"] == "assistant":
            return message["content"].strip()
    return ""


sft_sample = sft_train_raw[0]
dpo_sample = dpo_train_raw[0]

print("Raw SFT Prompt:\n")
print(sft_sample["prompt"])
print("\nRaw SFT Chosen Answer:\n")
print(get_last_assistant_text(sft_sample["chosen"]))

print("\n" + "=" * 80 + "\n")

print("Raw DPO Prompt:\n")
print(dpo_sample["prompt"])
print("\nRaw DPO Chosen:\n")
print(get_last_assistant_text(dpo_sample["chosen"]))
print("\nRaw DPO Rejected:\n")
print(get_last_assistant_text(dpo_sample["rejected"]))

Raw SFT Prompt:

Do you know something about crystallography and structure factor?

Raw SFT Chosen Answer:

Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.

The structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.

The structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.

Crystallography is a powerful tool for understanding the structure of materials. It has been used to determine the structures of many important materials, including metals, semiconductors, and pharmaceuticals. It is also used to study the structure of biological materials, such as proteins an

### Build SFT and DPO formatting functions

In [8]:
def make_sft_example(example):
    prompt = "User: " + example["prompt"].strip() + "\nAssistant"
    completion = " " + get_last_assistant_text(example["chosen"])
    return {
        "prompt" : prompt,
        "completion" : completion
    }


def make_dpo_example(example):
    prompt = "User: " + example["prompt"].strip() + "\nAssistant"
    chosen = " " + get_last_assistant_text(example["chosen"])
    rejected = " " + get_last_assistant_text(example["rejected"])
    return {
        "prompt" : prompt,
        "chosen": chosen,
        "rejected": rejected
    }

_Convert raw datasets into trainer-ready datasets_

In [9]:
sft_train = sft_train_raw.map(make_sft_example, remove_columns=sft_train_raw.column_names)
sft_val = sft_val_raw.map(make_sft_example, remove_columns=sft_val_raw.column_names)

dpo_train = dpo_train_raw.map(make_dpo_example, remove_columns=dpo_train_raw.column_names)
dpo_val = dpo_val_raw.map(make_dpo_example, remove_columns=dpo_val_raw.column_names)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [11]:
print(sft_train[0], "\n")
print(dpo_train[0])

{'prompt': 'User: Do you know something about crystallography and structure factor?\nAssistant', 'completion': ' Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.\n\nThe structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.\n\nThe structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.\n\nCrystallography is a powerful tool for understanding the structure of materials. It has been used to determine the structures of many important materials, including metals, semiconductors, and pharmaceuticals. It is also used to study the structure of biological materials, such as 

### Load tokenizer

In [12]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

In [13]:
print("Pad Token:", tokenizer.pad_token)
print("EOS Token:", tokenizer.eos_token)
print("Pad Token ID:", tokenizer.pad_token_id)

Pad Token: <|endoftext|>
EOS Token: <|endoftext|>
Pad Token ID: 0


### Load base causal language model

In [15]:
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
)

base_model.config.use_cache = False
base_model.enable_input_require_grads()

print(base_model.__class__.__name__)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM


### Add LoRA adapters for SFT stage

In [16]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

sft_model = get_peft_model(base_model, peft_config)
sft_model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 136,358,208 || trainable%: 1.3517


### Configure SFT training

In [17]:
sft_args = SFTConfig(
    output_dir="smollm2_sft_output",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

### Run supervised fine-tuning

In [18]:
sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    processing_class=tokenizer
)

sft_trainer.train()

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss
100,1.721338,1.754847


TrainOutput(global_step=188, training_loss=1.7611874823874616, metrics={'train_runtime': 161.2947, 'train_samples_per_second': 18.6, 'train_steps_per_second': 1.166, 'total_flos': 497878990848000.0, 'train_loss': 1.7611874823874616})

_Save SFT adapter_

In [19]:
sft_trainer.model.save_pretrained("smollm2_sft_adapter")
tokenizer.save_pretrained("smollm2_sft_adapter")

('smollm2_sft_adapter/tokenizer_config.json',
 'smollm2_sft_adapter/tokenizer.json')

### Configure DPO training

In [20]:
dpo_args = DPOConfig(
    output_dir="smollm2_dpo_output",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    max_length=MAX_LENGTH,
    beta=0.1,
    report_to="none",
    fp16=torch.cuda.is_available()
)

### Run direct preference optimization

In [21]:
dpo_trainer = DPOTrainer(
    model=sft_trainer.model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    processing_class=tokenizer

)

dpo_trainer.train()

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
50,0.688935,0.688103
100,0.687234,0.685185
150,0.678608,0.683172


TrainOutput(global_step=188, training_loss=0.6855030389542275, metrics={'train_runtime': 407.3244, 'train_samples_per_second': 7.365, 'train_steps_per_second': 0.462, 'total_flos': 1012555023224832.0, 'train_loss': 0.6855030389542275})

_Save DPO adapter_

In [22]:
dpo_trainer.model.save_pretrained("smollm2_dpo_adapter")
tokenizer.save_pretrained("smollm2_dpo_adapter")

('smollm2_dpo_adapter/tokenizer_config.json',
 'smollm2_dpo_adapter/tokenizer.json')

### Load saved SFT and DPO adapters for comparison

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
)

dpo_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype
)

sft_eval_model = PeftModel.from_pretrained(sft_base_model, "smollm2_sft_adapter").to(device)
dpo_eval_model = PeftModel.from_pretrained(dpo_base_model, "smollm2_dpo_adapter").to(device)

print(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

cuda


### Build generation function

In [24]:
def generate_response(model, prompt):
    model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text.strip()

_Compare SFT and DPO on a custom prompt_

In [25]:
custom_prompt = "User: Explain overfitting in simple words with one small example.\nAssistant:"

print("Prompt:\n")
print(custom_prompt)

print("\nSFT Output:\n")
print(generate_response(sft_eval_model, custom_prompt))

print("\nDPO Output:\n")
print(generate_response(dpo_eval_model, custom_prompt))

Prompt:

User: Explain overfitting in simple words with one small example.
Assistant:

SFT Output:

Overfitting occurs when a model is too complex or has too many features, which makes it difficult to learn from the data. This is why it is important to train the model on a smaller dataset and fine-tune it on the larger dataset.

Let's say you have a dataset with 100 features and a model with 100 features. Overfitting can occur when the model is too complex or has too many features, which makes it difficult to learn from the data. This is why it is important to train the model on a smaller dataset and fine-tune

DPO Output:

Overfitting occurs when a model is too complex or has too many features, which makes it difficult to learn from the data. This can lead to overfitting, which is a significant issue in machine learning.

Let's consider an example to understand overfitting better. Imagine you are a machine learning model trained on a dataset with a large number of features. The model 

_Compare SFT and DPO on one real preference sample_

In [26]:
pref_sample = dpo_val[0]

print("Prompt:\n")
print(pref_sample["prompt"])

print("\nChosen Answer:\n")
print(pref_sample["chosen"])

print("\nRejected Answer:\n")
print(pref_sample["rejected"])

print("\nSFT Output:\n")
print(generate_response(sft_eval_model, pref_sample["prompt"]))

print("\nDPO Output:\n")
print(generate_response(dpo_eval_model, pref_sample["prompt"]))

Prompt:

User: In this task, you are given a second sentence. Your task is to generate the first sentence on the same topic but incoherent and inconsistent with the second sentence.

Q: Additionally , some groups may contain other specialists , such as a heavy weapons or language expert .

A: Each squad member is specially trained as a weapons expert , medic , combat engineer or communications expert , respectively .
****
Q: However , the General Accounting Office identified 125 countries that received U.S. training and assistance for their police forces during fiscal year 1990 at a cost of at least $117 million .

A: No government agency is in charge of calculating the cost .
****
Q: But his frozen body was found in the ice in Charlotte ( Rochester ) early the next spring by Silas Hudson .

A:
Assistant

Chosen Answer:

 Could you provide some context or information about what you are looking for or any particular questions you have, so I can assist better?

Rejected Answer:

 As an A